# 情報科学実験II
知能ロボティクス(H1)<br>
担当：小林邦和<br>

更新履歴<br>
2023/4/20 ver1.0<br>
2024/4/11 ver1.1(ネットワーク構成は全結合層2層)

# 1.実験準備

本コードは以下のサイト「Deep Learningで犬・猫を分類してみよう」を参考にしています． <br>
https://aiacademy.jp/texts/show/?id=164

ただし，プログラムコードを一部改変，コメントも一部変更・追記しています．

画像をアップロードする方法にgoogle driveをマウントして取得する方法を利用しています
（この方法の場合，毎回google driveの接続承認を行う必要あり）．

Google ColabでGPUを使えるようにする方法

1.   画面上部にあるランタイムをクリック
2.   「ランタイムのタイプを変更」をクリック
3.   ハードウェアアクセラレータ → GPU
4.   保存

## 1-1 ライブラリの読み込み

In [ ]:
import numpy as np
import copy

# 画像操作用ライブラリ
import glob
from PIL import Image,ImageFile

# ニューラルネットワーク用ライブラリ
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Conv2D,MaxPooling2D,Activation,Dropout,Flatten,Dense,BatchNormalization
from keras.optimizers import RMSprop
from keras import utils,regularizers
from keras.models import load_model
from keras.utils import plot_model

# 機械学習用ライブラリ
from sklearn.metrics import confusion_matrix,accuracy_score

# データ可視化用ライブラリ
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import SVG

# GPUに接続されているかのチェック
# /device:GPU:0 と表示されていれば接続OK
if str(get_ipython()).startswith("<google.colab."):
    print(tf.test.gpu_device_name())

# Google driveのマウント
if str(get_ipython()).startswith("<google.colab."):
    from google.colab import drive
    drive.mount('/content/drive')

# パス設定(適宜各自のフォルダ構成へ変更)
if str(get_ipython()).startswith("<google.colab."):
    base_path = '/content/drive/My Drive/各自のフォルダ構成へ変更/' #Google Colab環境の場合
else:
    base_path = 'C:/Users/各自のフォルダ構成へ変更/' # ローカル環境(Windows)の場合
#    base_path = '/Users/各自のフォルダ構成へ変更/' # ローカル環境(macOS)の場合

## 1-2 データクローリング

In [ ]:
# クローリング(icrawler)による画像収集方法
# 詳しい引数については以下のURLを参照
# https://icrawler.readthedocs.io/en/latest/
#
# 本実験では画像を配付するためコメントアウト

"""
!pip install icrawler

from icrawler.builtin import GoogleImageCrawler
from IPython.display import IPImage,display_jpeg

# "dog_image_dir"がdirectory名となる
crawler = GoogleImageCrawler(storage={"root_dir": "dog_image_dir"})

# keywordで検索ワードを設定
# max_numで設定しても必ずその枚数集まるとは限らない(以下のようなエラーが出る)
# ERROR:downloader:Response status code 400
crawler.crawl(keyword="dog", max_num=100)

# スクレイピングした猫画像の一例の表示
display_jpeg(IPImage("./dog_image_dir/000001.jpg"))
"""

## 1-3 犬の画像データの読み込み

In [ ]:
# 犬画像の保存場所(適宜各自のフォルダ構成へ変更)
dog_path = base_path + 'dataset/dog_train/'

# globを用いてファイルを取得
dog_files = glob.glob(dog_path + '*')

# 犬画像の一例表示
img = mpimg.imread(dog_files[0])
imgplot = plt.imshow(img)

## 1-4 猫の画像データの読み込み

In [ ]:
# 猫画像の保存場所(適宜各自のフォルダ構成へ変更)
cat_path = base_path + 'dataset/cat_train/'

# globを用いてファイルを取得
cat_files = glob.glob(cat_path + '*')

# 猫画像の一例表示
img = mpimg.imread(cat_files[0])
imgplot = plt.imshow(img)

# 2.データの整形と学習データの作成

In [ ]:
# IOError: image file is truncated (0 bytes not processed)回避のため
# 注意：Trueにするとファイルが壊れていても読み込むため扱うデータに破損がないか確認
ImageFile.LOAD_TRUNCATED_IMAGES = True

# indexを教師ラベルとして割り当てるため、0にはdogを指定し、1には猫を指定
classes = ["dog", "cat"]
# len：listの長さを返す
num_classes = len(classes)

class_path = [dog_path, cat_path]

image_size = 64 # リサイズする画像サイズの設定(64x64)

X_train = [] # 訓練用データの画像
y_train = [] # 訓練用データのクラスラベル

for index, classlabel in enumerate(classes):
    """
    # スクレイピングをしている場合
    photos_dir = "./" + classlabel + "_image_dir"
    files = glob.glob(photos_dir + "/*")
    """

    photos_dir = class_path[index] # 画像フォルダの指定
    files = glob.glob(photos_dir + '*') # 画像ファイルの指定

    for i, file in enumerate(files):
        image = Image.open(file) # 画像の読み込み
        image = image.convert("RGB") # pngの場合RGB以外にもFGBAなどでエンコードされている場合があるためRGBモードへ変換
        image = image.resize((image_size, image_size)) # 画像リサイズ(64x64)

        data = np.asarray(image) # imageをNumPy配列に変換
        # 学習データの水増し
        for angle in range(-20, 20, 5):
            # 5度ずつ-20から20度まで回転させた画像をtrainに保存
            img_r = image.rotate(angle)
            data = np.asarray(img_r)
            X_train.append(data)
            y_train.append(index)
            # FLIP_LEFT_RIGHT　は 左右反転
            img_trains = img_r.transpose(Image.FLIP_LEFT_RIGHT)
            data = np.asarray(img_trains)
            X_train.append(data)
            y_train.append(index)

# trainデータをnumpy.ndarrayに
X_train = np.array(X_train)
y_train = np.array(y_train)

# 3.ニューラルネットワークモデルの構築と学習

In [ ]:
# データ読み込みの関数
def load_data(X_train,y_train):
    X_train = X_train.astype("float") / 255 # 入力データの各画素値を0-1の範囲で正規化
    y_train = utils.to_categorical(y_train, num_classes) # ラベル情報のone hot vector化
    return X_train, y_train

#モデル訓練の関数
def train(X, y):
    # Sequential: ニューラルネットワークを構築するためのモデルクラス
    model = Sequential()

    # Flattenレイヤー: 多次元配列を1次元配列に変換するレイヤー
    model.add(Flatten()) # 引数なし
    # Denseレイヤー： 全結合層を定義するためのレイヤー
    model.add(Dense(512)) # 512個のニューロンを持つ全結合層
    model.add(Activation('sigmoid')) # 活性化関数はシグモイド関数
    model.add(Dense(2)) # 2クラス分類のためユニット数は2を指定
    model.add(Activation('softmax')) # 活性化関数はSoftmax関数

    # モデルのコンパイル
    # 引数例
    # optimizer: 最適化アルゴリズムを指定(adam, RMSprop, sgd, etc)
    # loss: 損失関数を指定(categorical_crossentropy, mean_squared_error, etc)
    # metrics: モデルの性能評価を行う関数(accuracy, precision, recall, f1score)
    # loss_weights: 各出力に対する損失の重み
    model.compile(
        optimizer = 'sgd',
        loss = 'mean_squared_error',
        metrics = ['accuracy']
        )

    # モデルにtrainデータをfit
    # batch_size: 1回の学習に用いるデータの数, epochs: エポック数
    model.fit(X, y, batch_size=28, epochs=20)
    # Keras形式でNNモデルを保存
    model.save(base_path + 'CNN_dog_cat.keras')

    return model

# メイン関数： データの読み込みとモデルの学習
def main(X_train, y_train):
    # 正規化とワンホットベクター化
    X_train, y_train = load_data(X_train, y_train)

    # モデルの学習
    model = train(X_train, y_train)

# メイン関数の実行
main(X_train, y_train)

# 4.モデルの性能評価

## 4-1 検証データによる評価
ネットワーク構成の見直しやハイパーパラメータの最適化のため

In [ ]:
# 犬画像の指定(適宜各自のフォルダ構成へ変更)
X_val_dog = glob.glob(base_path + 'dataset/dog_val/*') # 検証データ

# 猫画像の指定(適宜各自のフォルダ構成へ変更)
X_val_cat = glob.glob(base_path + 'dataset/cat_val/*') # 検証データ

#　モデルパラメータファイルの指定
keras_param = base_path + 'CNN_dog_cat.keras'

# 画像サイズの指定
imsize = (64, 64)

# 画像読み込みの関数
def load_image(path):
    img = Image.open(path)
    img = img.convert('RGB')
    # (64, 64)にリサイズ
    img = img.resize(imsize)
    # 画像データをnumpyに変換
    img = np.asarray(img)
    img = img / 255.0
    return img

# モデルパラメータの読み込み
model = load_model(keras_param)

# 犬画像の予測
pred_dog = []
for i in range(len(X_val_dog)):
    img = load_image(X_val_dog[i])  # 画像の読み込みと整形
    pred = model.predict(np.array([img])) #　クラス確率の計算
    pred = np.argmax(pred, axis=1) # クラス確率の最大値が予測結果
    pred_dog.append(int(pred)) # 予測ラベルの追加

# 猫画像の予測
pred_cat = []
for i in range(len(X_val_cat)):
    img = load_image(X_val_cat[i])  # 画像の読み込みと整形
    pred = model.predict(np.array([img]))
    pred = np.argmax(pred, axis=1)
    pred_cat.append(int(pred))

# 正解ラベル
y_test = []
y_test = [0 for _ in range(len(X_val_dog))]
y_test.extend([1 for _ in range(len(X_val_cat))])

# 予測ラベル
y_pred = []
y_pred = copy.deepcopy(pred_dog)
y_pred.extend(pred_cat)

# 混同行列
cm = confusion_matrix(y_test, y_pred)
print("混同行列:")
print(cm)

# 正解率
print("正解率：",accuracy_score(y_test,y_pred))

## 4-2 テストデータによる評価
最終的な評価結果

In [ ]:
# 後日配付するテーストデータ入手後に以下のコメント解除
"""
# 犬画像の指定(適宜各自のフォルダ構成へ変更)
X_test_dog = glob.glob(base_path + 'dataset/dog_test1/*') # テストデータ(第1期)
#X_test_dog = glob.glob(base_path + 'dataset/dog_test2/*') # テストデータ(第2期)

# 猫画像の指定(適宜各自のフォルダ構成へ変更)
X_test_cat = glob.glob(base_path + 'dataset/cat_test1/*') # テストデータ(第1期)
#X_test_cat = glob.glob(base_path + 'dataset/cat_test2/*') # テストデータ(第2期)

# モデルパラメータの読み込み
model = load_model(keras_param)

# 犬画像の予測
pred_dog = []
for i in range(len(X_test_dog)):
    img = load_image(X_test_dog[i])  # 画像の読み込みと整形
    pred = model.predict(np.array([img]))
    pred = np.argmax(pred, axis=1)
    pred_dog.append(int(pred))

# 猫画像の予測
pred_cat = []
model = load_model(keras_param)  # モデルの読み込み
for i in range(len(X_test_cat)):
    img = load_image(X_test_cat[i])  # 画像の読み込みと整形
    pred = model.predict(np.array([img]))
    pred = np.argmax(pred, axis=1)
    pred_cat.append(int(pred))

# 正解ラベル
y_test = []
y_test = [0 for _ in range(len(X_test_dog))]
y_test.extend([1 for _ in range(len(X_test_cat))])

# 予測ラベル
y_pred = []
y_pred = copy.deepcopy(pred_dog)
y_pred.extend(pred_cat)

# 混同行列
cm = confusion_matrix(y_test, y_pred)
print("混同行列:")
print(cm)

# 正解率
print("正解率：",accuracy_score(y_test,y_pred))
"""

# 5.付録

## 5-1 犬・猫のクラス確率の計算とクラス推定

In [ ]:

#　適当な画像の指定
testpic = glob.glob(base_path + 'images/*.jpg')

# 画像サイズの指定
imsize = (64, 64)

# modelの読み込み
model = load_model(keras_param)

pred_res = []
for i in range(len(testpic)):
    # 対象画像の読み込みと正規化
    img = load_image(testpic[i])
    # 元画像の表示
    img_orig = mpimg.imread(testpic[i])
    imgplot = plt.imshow(img_orig)
    plt.show()
    # 犬・猫のクラス確率の計算
    pred = model.predict(np.array([img]))
    print(pred)
    # クラス推定の結果
    pred_label = np.argmax(pred, axis=1)
    if pred_label == 0:
        print(">>> 推定結果→犬", flush=True)
    elif pred_label == 1:
        print(">>> 推定結果→猫", flush=True)

## 5-2 モデル図の描画

In [ ]:
#　モデルパラメータの読み込み
keras_param = base_path + 'CNN_dog_cat.keras'
model = load_model(keras_param)  # modelの読み込み

# モデル図の保存(png形式)
plot_model(model, show_shapes=True, show_layer_activations=True, dpi=100, to_file=base_path + 'model.png')